<a id="top"></a>

# Demo: state and reducers -- the message history, now typed

```yaml
title: "Demo: state and reducers -- the message history, now typed"
keywords: langgraph, state, reducer, add_messages, message history, tool_use, tool_result, invariant, overwrite, recursion limit, graphrecursionerror, state boundary, teacher demo
estimated duration: 16
```

> **Lesson:** L14. Teacher demo -- Demo 3 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L14/demos_or_activities.md).
> Runs **fully offline** -- a deterministic `StubChat` stands in for the model and the REAL
> `ToolNode` runs the REAL tools, so the **reducer break** is reproducible, free, and keyless. (The
> live agent was L1403; reducers are about LangGraph's *state merging*, not the model.)

## Contents

- [1. Setup -- the agent, parameterized by its state schema](#1-setup----the-agent-parameterized-by-its-state-schema)
- [2. State is the message history -- watch it grow](#2-state-is-the-message-history----watch-it-grow)
- [3. A node returns an update; a reducer merges it](#3-a-node-returns-an-update-a-reducer-merges-it)
- [4. Break the reducer on purpose](#4-break-the-reducer-on-purpose)
- [5. What belongs in state -- and what doesn't](#5-what-belongs-in-state----and-what-doesnt)
- [6. Takeaways](#6-takeaways)

## 1. Setup -- the agent, parameterized by its state schema

Given: the shared tools, the offline `StubChat` (scripted `calculator` -> `lookup` -> answer), and
a `build_agent(state_cls)` helper that wires the *same* shallow agent over whatever **state schema**
you hand it. That parameter is the whole demo: we'll build it once with the **right** `messages`
reducer and once with a **broken** one, and change nothing else.

In [1]:
from typing import Annotated, Any, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.tools import StructuredTool
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from fluffy_potato_curriculum.common import tools as T

calculator = StructuredTool.from_function(T.calculator, name="calculator", description="arithmetic")
lookup = StructuredTool.from_function(T.lookup, name="lookup", description="city population")
LC_TOOLS = [calculator, lookup]


class StubChat:
    """Offline, scripted stand-in: turn 0 -> calculator, turn 1 -> lookup, then a final answer."""

    def bind_tools(self, tools: list[object]) -> "StubChat":
        return self

    def invoke(self, messages: list[BaseMessage]) -> AIMessage:
        ai_turns = sum(1 for m in messages if isinstance(m, AIMessage))
        if ai_turns == 0:
            return AIMessage(
                content="",
                tool_calls=[{"name": "calculator", "args": {"expression": "21*2"}, "id": "c1"}],
            )
        if ai_turns == 1:
            return AIMessage(
                content="", tool_calls=[{"name": "lookup", "args": {"city": "Tokyo"}, "id": "l1"}]
            )
        return AIMessage(content="21*2 = 42, and Tokyo's population is 37000000.")


model = StubChat()
TASK = "Use the calculator to compute 21 * 2, then tell me the population of Tokyo."


def build_agent(state_cls: type) -> Any:
    """Wire the SAME shallow agent over whatever state schema is passed. The model client and the
    tools are DEPENDENCIES closed over here -- not part of the state (see section 5)."""

    def agent_node(state: Any) -> dict[str, object]:
        return {
            "messages": [model.bind_tools(LC_TOOLS).invoke(state["messages"])],
            "step": state["step"] + 1,
        }

    def route(state: Any) -> str:
        last = state["messages"][-1]
        return "tools" if isinstance(last, AIMessage) and last.tool_calls else END

    b = StateGraph(state_cls)
    b.add_node("agent", agent_node)
    b.add_node("tools", ToolNode(LC_TOOLS, handle_tool_errors=True))
    b.set_entry_point("agent")
    b.add_conditional_edges("agent", route, {"tools": "tools", END: END})
    b.add_edge("tools", "agent")
    return b.compile()

[↑ Back to top](#top)

## 2. State is the message history -- watch it grow

Build the agent with the **correct** state (`messages` uses the `add_messages` reducer) and invoke
it. Print the message list: it's the *same* growing history L10 mutated in place -- user, then
assistant-with-`tool_use`, then tool-result, then assistant... -- now a typed value LangGraph
threads through every node.

In [2]:
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]  # APPEND reducer
    step: int


good_app = build_agent(AgentState)
final = good_app.invoke({"messages": [HumanMessage(TASK)], "step": 0})

for m in final["messages"]:
    tools_asked = [c["name"] for c in m.tool_calls] if isinstance(m, AIMessage) else []
    print(f"{type(m).__name__:12} tool_calls={tools_asked} content={str(m.content)[:45]!r}")
print("\nsteps:", final["step"])

HumanMessage tool_calls=[] content='Use the calculator to compute 21 * 2, then te'
AIMessage    tool_calls=['calculator'] content=''
ToolMessage  tool_calls=[] content='42'
AIMessage    tool_calls=['lookup'] content=''
ToolMessage  tool_calls=[] content='37000000'
AIMessage    tool_calls=[] content="21*2 = 42, and Tokyo's population is 37000000"

steps: 3


[↑ Back to top](#top)

## 3. A node returns an update; a reducer merges it

The trap is to think a node *sets* the state. It doesn't -- a node **returns an update** that
LangGraph **merges** into state via a per-field **reducer**:

- `messages` uses **`add_messages`**, which **appends**. Each `agent` node returns
  `{"messages": [response]}` and the reducer *adds* it -- it never replaces the list.
- `step` uses the **default reducer**, which **overwrites** (last write wins).

So the assistant `tool_use` turn and the following `tool_result` turn **accumulate in order, by
construction.** That ordering is the L10 invariant -- the one you used to maintain by hand -- now
enforced by the reducer.

In [3]:
# Proof that messages APPENDS (grows) while step OVERWRITES (one value):
print("messages is a growing list of length:", len(final["messages"]))
print("step is a single overwritten int    :", final["step"])

messages is a growing list of length: 6
step is a single overwritten int    : 3


[↑ Back to top](#top)

## 4. Break the reducer on purpose

Now swap `Annotated[list, add_messages]` for a **plain `list[BaseMessage]`** -- the **default
overwrite** reducer. Change *nothing else* and re-run.

Each node's `{"messages": [response]}` now **clobbers** the whole history instead of appending. The
prior `tool_use` turn vanishes before its `tool_result` can pair with it; the agent loses the
thread, re-issues the same call, and -- with no stop condition it can reach -- **hits LangGraph's
recursion limit**. *This is the single most common L10 bug, reborn in graph form.*

In [4]:
class BrokenState(TypedDict):
    messages: list[BaseMessage]  # NO add_messages -> the DEFAULT reducer OVERWRITES (the bug)
    step: int


broken_app = build_agent(BrokenState)
try:
    # recursion_limit is the framework's max_steps -- we set it low so the bug surfaces fast.
    broken_app.invoke({"messages": [HumanMessage(TASK)], "step": 0}, {"recursion_limit": 8})
    print("(unexpected: the broken graph terminated)")
except GraphRecursionError as exc:
    print("GraphRecursionError:", str(exc).split(".")[0])
    print("\n-> history was clobbered each turn; the tool_use/tool_result pairing broke,")
    print("   so the loop never reached END and hit the recursion cap (L10's max_steps).")

GraphRecursionError: Recursion limit of 8 reached without hitting a stop condition

-> history was clobbered each turn; the tool_use/tool_result pairing broke,
   so the loop never reached END and hit the recursion cap (L10's max_steps).


[↑ Back to top](#top)

## 5. What belongs in state -- and what doesn't

The break also clarifies the **state boundary**:

- **In state** -- data that *flows between nodes and must persist across the loop*: the `messages`,
  the `step` counter.
- **Not in state** -- the **model client** (`StubChat`/`ChatAnthropic`) and the **tool registry**
  (`LC_TOOLS`). Those are **dependencies** wired in at build time -- in `build_agent` they're closed
  over by the nodes, not threaded through `invoke`.

Conflating the two is how a graph gets tangled. A shallow agent's state stays small: **messages
plus one counter.** Richer state is L15's (patterns) and L19's (context management) job.

In [5]:
# The dependencies live OUTSIDE the state -- the initial state carries only data, not the client:
initial_state = {"messages": [HumanMessage(TASK)], "step": 0}
print("keys we invoke with:", list(initial_state))  # ['messages', 'step'] -- no model, no tools

keys we invoke with: ['messages', 'step']


[↑ Back to top](#top)

## 6. Takeaways

- **State is the message history, now typed and framework-managed.** It's the same list L10
  mutated; LangGraph threads it through every node.
- **A node returns an update; a reducer merges it.** "Return a dict and hope" is the trap -- learn
  the reducer, not just the return.
- **`add_messages` enforces the L10 invariant by construction.** Append, don't overwrite -- that's
  what keeps every `tool_use` paired with its `tool_result`.
- **Break the reducer and the L10 bug returns** -- as a clobbered history that loops to the
  recursion cap. The step cap didn't go away; it caught the runaway.
- **State is data that flows; clients and tools are dependencies.** Keep state to messages + one
  counter. Next: carry the eval set across + find the trace in Langfuse ([L1407](L1407_lecture.ipynb)).

[↑ Back to top](#top)